In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8')

df = pd.read_csv('../data/ai4i2020.csv')

sensor_cols = ['Air temperature [K]', 'Process temperature [K]', 
               'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

# Rolling features
window_size = 10
for col in sensor_cols:
    df[f'{col}_roll_mean'] = df[col].rolling(window=window_size).mean()
    df[f'{col}_roll_std']  = df[col].rolling(window=window_size).std()
    df[f'{col}_roll_var']  = df[col].rolling(window=window_size).var()

# Domain features
df['temp_delta'] = (
    df['Process temperature [K]'] 
    - df['Air temperature [K]']
)

df['power'] = (
    df['Torque [Nm]'] 
    * df['Rotational speed [rpm]']
)

df['tool_wear_rate'] = (
    df['Tool wear [min]'] 
    / (df['Rotational speed [rpm]'] + 1)
)

df['torque_per_rpm'] = (
    df['Torque [Nm]'] 
    / (df['Rotational speed [rpm]'] + 1)
)

df['temp_wear_interaction'] = (
    df['Process temperature [K]'] 
    * df['Tool wear [min]']
)

# Encode type
df['type_encoded'] = df['Type'].map({
    'L': 0,
    'M': 1,
    'H': 2
})

# Drop unnecessary cols
cols_to_drop = [
    'UDI',
    'Product ID',
    'Type',
    'TWF',
    'HDF',
    'PWF',
    'OSF',
    'RNF'
]

df = df.drop(columns=cols_to_drop)
df = df.dropna().reset_index(drop=True)

print(f"✅ Full pipeline rebuilt | Shape: {df.shape}")

In [ ]:
# Lag features capture "what happened before"
# Useful for time-series failure prediction

lag_cols = [
    'Torque [Nm]',
    'Tool wear [min]',
    'Rotational speed [rpm]',
    'power'
]

for col in lag_cols:
    df[f'{col}_lag1'] = df[col].shift(1)
    df[f'{col}_lag2'] = df[col].shift(2)
    df[f'{col}_lag3'] = df[col].shift(3)

df = df.dropna().reset_index(drop=True)

print(f"✅ Lag features added | Shape: {df.shape}")

print("\nLag features created:")
lag_features = [c for c in df.columns if 'lag' in c]

for f in lag_features:
    print(f"  → {f}")

In [ ]:
# Trend features (rate of change)

trend_cols = [
    'Torque [Nm]',
    'Tool wear [min]',
    'Rotational speed [rpm]',
    'power'
]

for col in trend_cols:
    df[f'{col}_diff1'] = df[col].diff(1)
    df[f'{col}_pct_change'] = df[col].pct_change()

# Clean NaNs created by diff/pct_change
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna().reset_index(drop=True)

print(f"✅ Trend features added | Shape: {df.shape}")

trend_features = [
    c for c in df.columns
    if 'diff1' in c or 'pct_change' in c
]

print("\nTrend features created:")
for feat in trend_features:
    print(f" → {feat}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
axes = axes.flatten()

plot_cols = [
    'Torque [Nm]_lag1',
    'Tool wear [min]_lag1',
    'power_lag1',
    'Rotational speed [rpm]_lag1'
]

for i, col in enumerate(plot_cols):
    axes[i].hist(
        df[df['Machine failure'] == 0][col],
        bins=50,
        alpha=0.6,
        color='steelblue',
        label='No Failure'
    )

    axes[i].hist(
        df[df['Machine failure'] == 1][col],
        bins=50,
        alpha=0.8,
        color='red',
        label='Failure'
    )

    axes[i].set_title(col)
    axes[i].legend()

plt.suptitle(
    'Lag Features: Failure vs No Failure',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.savefig('../src/day4_lag_features.png')
plt.show()

print("✅ Lag feature visualization saved!")

In [ ]:
# Correlation of all numeric features with target

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()

numeric_cols.remove('Machine failure')

corr_target = (
    df[numeric_cols + ['Machine failure']]
    .corr()['Machine failure']
    .drop('Machine failure')
    .sort_values(key=abs, ascending=False)
)

print("=== Top 20 Features Correlated with Failure ===")
print(corr_target.head(20).to_string())

# Plot
plt.figure(figsize=(10, 8))

corr_target.head(20).plot(
    kind='barh',
    color='coral',
    edgecolor='black'
)

plt.title(
    'Top 20 Features Correlated with Failure',
    fontsize=13,
    fontweight='bold'
)

plt.xlabel('Correlation')
plt.axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('../src/day4_feature_importance.png')
plt.show()

print("✅ Feature correlation chart saved!")

In [ ]:
# Outlier detection using Z-score

sensor_check_cols = [
    'Torque [Nm]',
    'Rotational speed [rpm]',
    'Tool wear [min]',
    'power'
]

outlier_summary = {}

for col in sensor_check_cols:
    z_scores = np.abs(stats.zscore(df[col]))
    outliers = (z_scores > 3).sum()

    outlier_summary[col] = outliers

print("=== Outlier Detection Summary ===")
for k, v in outlier_summary.items():
    print(f"{k}: {v} outliers")

# Visualize
plt.figure(figsize=(8, 5))

plt.bar(
    outlier_summary.keys(),
    outlier_summary.values()
)

plt.title(
    'Outlier Count by Feature',
    fontsize=13,
    fontweight='bold'
)

plt.ylabel('Number of Outliers')
plt.xticks(rotation=20)

plt.tight_layout()
plt.savefig('../src/day4_outliers.png')
plt.show()

print("✅ Outlier analysis saved!")

In [ ]:
# Save final engineered dataset

df.to_csv('../data/week1_final_dataset.csv', index=False)

print("✅ Final Week 1 dataset saved!")
print(f"Dataset shape: {df.shape}")
print(f"Total columns/features: {len(df.columns)}")
print("\nSaved path:")
print("../data/week1_final_dataset.csv")